# Vidya Graph POC · SmolLM2-135M
Test chart questions from data tables and generate symbol plots for bar, line, and circular pie charts.

Runs on a CPU-only laptop (no GPU needed). If a GPU is available it is used automatically. On CPU, expect training and scoring to take noticeably longer. Run all cells in order. The first run downloads the public model and datasets. You only need to upload this notebook. Compare the before-and-after scores. The notebook writes before.json, after.json, and graph_lora.zip under graph_poc_smol135m (in /kaggle/working on Kaggle, otherwise in the notebook folder). The default run uses 384 training rows, two epochs, and 54 held-out examples. Input is a chart table in text. Chart answers are scored by normalized exact match. Plot requests are scored by full spec match, with valid spec counts reported independently.

[SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) · [ChartQA tables](https://huggingface.co/datasets/nmayorga7/chartqa-tables) · [Original ChartQA questions](https://github.com/vis-nlp/ChartQA)


## 1. Install dependencies

Enable Internet for the initial package, dataset, and model downloads. Skip this cell if the listed packages are already installed. Restart the kernel if you imported the old versions before running the installer.

If PEFT reports the Kaggle TorchAO version conflict, run `%pip uninstall -y torchao` in a separate cell, restart the kernel, and resume at Step 2.


In [ ]:
%pip install -q "transformers==5.23.0" "peft==0.21.3" "huggingface-hub==1.33.0" "Pillow==12.3.0" pyarrow


## 2. Load the data and model


In [ ]:
import io
import json
import math
import random
import re
import shutil
from decimal import Decimal
from pathlib import Path
from string import ascii_uppercase
from urllib.request import urlopen

import pandas as pd
import torch
from peft import LoraConfig, get_peft_model
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
OUT = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()) / "graph_poc_smol135m"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Device:", torch.cuda.get_device_name(0) if USE_AMP else "CPU")
set_seed(42)

KINDS = ("bar", "line", "pie")
QA_SYSTEM = "Answer the question using the chart table. Return only the short answer."
PLOT_SYSTEM = (
    'Return one JSON object with this structure: {"type":"bar","title":"Title",'
    '"x":{"label":"Category","values":["A","B"]},'
    '"y":{"label":"Value","values":[1,2]}}. Use the requested type (bar, line, pie), '
    "title and axis labels. Copy all values in order. Use numeric x for line and "
    "category strings for bar or pie. Output only JSON."
)

def fetch(url):
    with urlopen(url, timeout=60) as response:
        return response.read()

def chartqa(split, count, blocked=()):
    root = "https://raw.githubusercontent.com/vis-nlp/ChartQA/main/ChartQA%20Dataset"
    hub = "https://huggingface.co/datasets/nmayorga7/chartqa-tables/resolve/main/data"
    qa = json.loads(fetch(f"{root}/{split}/{split}_human.json"))
    frame = pd.read_parquet(
        io.BytesIO(fetch(f"{hub}/{split}-00000-of-00001.parquet")),
        columns=["imgname", "table_csv"],
    ).dropna()
    tables = dict(zip(frame["imgname"], frame["table_csv"]))
    random.Random(42).shuffle(qa)
    seen_ids = {row["id"] for row in blocked}
    seen_tables = {row["table"] for row in blocked}
    rows = []
    for item in qa:
        name = item["imgname"]
        table = tables.get(name, "").strip()
        answer = item["label"]
        if (not table or len(table) > 950 or name in seen_ids or table in seen_tables
                or not isinstance(answer, str) or not answer.strip()):
            continue
        seen_ids.add(name)
        seen_tables.add(table)
        rows.append(dict(task="qa", id=name, table=table,
                         query=f"Chart table (CSV):\n{table}\n\nQuestion: {item['query']}",
                         target=answer))
        if len(rows) == count:
            return rows
    raise RuntimeError(f"Only found {len(rows)} eligible {split} examples.")

def plot_rows(count_per_type, seed, used):
    rng = random.Random(seed)
    rows = []
    for kind in KINDS:
        for _ in range(count_per_type):
            while True:
                size = rng.randint(3, 6)
                x = (sorted(rng.sample(range(15), size)) if kind == "line"
                     else rng.sample(list(ascii_uppercase[:12]), size))
                y = [rng.randint(1, 35) for _ in x]
                key = (tuple(x), tuple(y))
                if key not in used:
                    used.add(key)
                    break
            title = rng.choice(["Class survey", "Measurements", "Weekly results"])
            xlabel = "Time (s)" if kind == "line" else "Category"
            ylabel = "Value"
            expected = dict(type=kind, title=title, x=dict(label=xlabel, values=x),
                            y=dict(label=ylabel, values=y))
            table = "\n".join(f"{a},{b}" for a, b in zip(x, y))
            verb = rng.choice(["Draw", "Plot", "Create"])
            query = (f"{verb} a {kind} chart titled {title}.\n"
                     f"X label: {xlabel}\nY label: {ylabel}\n"
                     f"Use these x,y rows:\n{table}")
            rows.append(dict(task=kind, query=query, target=json.dumps(expected)))
    return rows

used = set()
test_qa = chartqa("test", 30)
train_rows = chartqa("train", 192, blocked=test_qa) + plot_rows(64, 17, used)
test_rows = test_qa + plot_rows(8, 23, used)
print(f"{len(train_rows)} training rows · {len(test_rows)} held-out rows")

tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
dtype = torch.float16 if USE_AMP else torch.float32
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(DEVICE)


## 3. Score the base model


In [ ]:
def prompt(row):
    system = QA_SYSTEM if row["task"] == "qa" else PLOT_SYSTEM
    return tok.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": row["query"]}],
        tokenize=False, add_generation_prompt=True,
    )


def parse_spec(reply):
    reply = reply.strip()
    fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", reply, re.I | re.S)
    try:
        spec = json.loads(fence[1] if fence else reply)
        kind = spec["type"]
        x, y = spec["x"]["values"], spec["y"]["values"]
        if (kind not in KINDS or not isinstance(spec["title"], str)
                or any(not isinstance(spec[a]["label"], str) for a in ("x", "y"))
                or not isinstance(x, list) or not isinstance(y, list)
                or not 2 <= len(x) == len(y) <= 8):
            return None
        def is_number(v):
            return type(v) in (int, float) and -1e9 <= v <= 1e9
        if not all(is_number(v) for v in y):
            return None
        if kind == "line":
            if not all(is_number(v) for v in x) or any(a >= b for a, b in zip(x, x[1:])):
                return None
        elif not all(isinstance(v, str) and v.strip() for v in x):
            return None
        if kind == "pie" and (min(y) < 0 or sum(y) <= 0):
            return None
        return spec
    except (TypeError, ValueError, KeyError):
        return None

def normalize_answer(answer):
    value = " ".join(answer.strip().casefold().split()).rstrip(".")
    match = re.fullmatch(r"([-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\s*(%?)", value)
    if match:
        return Decimal(match[1].replace(",", "")), match[2]
    return value

@torch.inference_mode()
def generate(model, rows, max_tokens):
    model.eval()
    batch = tok([prompt(row) for row in rows], padding=True,
                add_special_tokens=False, return_tensors="pt").to(DEVICE)
    output = model.generate(
        **batch, do_sample=False, max_new_tokens=max_tokens, use_cache=True,
        pad_token_id=tok.pad_token_id,
    )
    return tok.batch_decode(output[:, batch.input_ids.shape[1]:], skip_special_tokens=True)

def evaluate(model, name):
    records = []
    for task in ("qa", *KINDS):
        rows = [row for row in test_rows if row["task"] == task]
        task_records = []
        for start in range(0, len(rows), 8):
            batch = rows[start:start+8]
            replies = generate(model, batch, 32 if task == "qa" else 160)
            for row, reply in zip(batch, replies):
                spec = parse_spec(reply) if task != "qa" else None
                if task == "qa":
                    correct = normalize_answer(reply) == normalize_answer(row["target"])
                else:
                    expected = json.loads(row["target"])
                    correct = spec == expected
                task_records.append(dict(task=task, query=row["query"], expected=row["target"],
                                         reply=reply, valid=spec is not None, correct=correct))
        count = len(task_records)
        hits = sum(r["correct"] for r in task_records)
        valid = sum(r["valid"] for r in task_records)
        extra = f" | valid JSON specs {valid}/{count}" if task != "qa" else ""
        print(f"{name:>6} | {task:4} | correct {hits}/{count} ({hits/count:.1%}){extra}")
        records.extend(task_records)
    (OUT / f"{name}.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    return records

before = evaluate(base, "before")


## 4. Train LoRA and score again


In [ ]:
def encode(row):
    context = tok.encode(prompt(row), add_special_tokens=False)
    answer = tok.encode(row["target"].strip(), add_special_tokens=False)
    ids = context + answer + [tok.eos_token_id]
    labels = [-100] * len(context) + answer + [tok.eos_token_id]
    return torch.tensor(ids), torch.tensor(labels)

def collate(rows):
    ids, labels = zip(*rows)
    return {
        "input_ids": pad_sequence(ids, batch_first=True, padding_value=tok.pad_token_id),
        "attention_mask": pad_sequence([torch.ones_like(x) for x in ids], batch_first=True),
        "labels": pad_sequence(labels, batch_first=True, padding_value=-100),
    }

model = get_peft_model(base, LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
))
model.print_trainable_parameters()
loader = DataLoader([encode(row) for row in train_rows], batch_size=4,
                    shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4,
)
scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

for epoch in range(2):
    model.train()
    running_loss = 0.0
    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            loss = model(**{key: value.to(DEVICE) for key, value in batch.items()},
                         use_cache=False).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    print(f"Epoch {epoch+1} mean loss: {running_loss/len(loader):.3f}")

model.save_pretrained(OUT / "adapter")
tok.save_pretrained(OUT / "adapter")
after = evaluate(model, "after")
summary = [
    {"task": task, **{
        stage: sum(r["correct"] for r in records if r["task"] == task)
        for stage, records in (("before", before), ("after", after))
    }, "total": sum(row["task"] == task for row in test_rows)}
    for task in ("qa", *KINDS)
]
print(pd.DataFrame(summary).to_string(index=False))
print(shutil.make_archive(str(OUT / "graph_lora"), "zip", root_dir=OUT / "adapter"))


In [ ]:
from os.path import relpath
from IPython.display import FileLink, display

for name in ("graph_lora.zip", "after.json"):
    display(FileLink(relpath(OUT / name)))


## 5. Display symbol plots and test fresh requests


In [ ]:
import math
from string import ascii_uppercase

def render(spec):
    kind = spec["type"]
    x = spec["x"]["values"]
    y = spec["y"]["values"]
    print(f"\n{spec['title']}")

    if kind == "bar":
        scale = max(y) or 1
        width = max(map(len, x))
        for label, value in zip(x, y):
            blocks = "█" * round(24 * value / scale)
            print(f"{label:>{width}} │{blocks} {value:g}")

    elif kind == "line":
        lo, hi = min(y), max(y)
        grid = [[" "] * 37 for _ in range(9)]
        points = [
            (round(36 * (a - x[0]) / (x[-1] - x[0])),
             round(8 * (hi - b) / (hi - lo)) if hi != lo else 4)
            for a, b in zip(x, y)
        ]
        for (a, b), (c, d) in zip(points, points[1:]):
            steps = max(abs(c - a), abs(d - b), 1)
            for i in range(1, steps):
                grid[round(b + (d - b) * i / steps)][round(a + (c - a) * i / steps)] = "·"
        for col, row in points:
            grid[row][col] = "●"
        print(spec["y"]["label"])
        for i, row in enumerate(grid):
            tick = f"{hi:g}" if i == 0 else f"{lo:g}" if i == 8 else ""
            print(f"{tick:>6} │{''.join(row).rstrip()}")
        print("       └" + "─" * 37)
        print(f"        {x[0]:<36g}{x[-1]:g}  {spec['x']['label']}")

    elif kind == "pie":
        total = sum(y)
        ends = [sum(y[:i + 1]) / total for i in range(len(y))]
        for row in range(-7, 8):
            line = []
            for col in range(-15, 16):
                a, b = col / 15, row / 7
                if a * a + b * b > 1:
                    line.append(" ")
                    continue
                angle = (math.atan2(a, -b) / math.tau) % 1
                i = next(i for i, end in enumerate(ends) if angle < end)
                line.append(ascii_uppercase[i])
            print("".join(line).rstrip())
        for letter, label, value in zip(ascii_uppercase, x, y):
            print(f"{letter}  {label}  {value / total:.1%}")


In [ ]:
tests = [
    (
        "bar", "Station rainfall", "Station", "Rainfall (mm)",
        ["North Gate", "Central Park", "South Pier"], [12.5, 8.25, 12.5],
    ),
    (
        "line", "Water level", "Time (min)", "Height (cm)",
        [0, 2, 7, 11], [3.5, 4.0, 7.25, 6.5],
    ),
    (
        "pie", "Weekly budget", "Category", "Amount",
        ["Food and cooking", "Bus fares", "School supplies"], [45.5, 25, 29.5],
    ),
]

for kind, title, x_label, y_label, x, y in tests:
    render({
        "type": kind,
        "title": title,
        "x": {"label": x_label, "values": x},
        "y": {"label": y_label, "values": y},
    })


In [ ]:
for kind, title, x_label, y_label, x, y in tests:
    expected = {
        "type": kind,
        "title": title,
        "x": {"label": x_label, "values": x},
        "y": {"label": y_label, "values": y},
    }
    table = "\n".join(f"{a},{b}" for a, b in zip(x, y))
    query = (
        f"Make a {kind} chart called {title} from these rows.\n"
        f"Use {x_label} for X and {y_label} for Y.\n{table}"
    )
    reply = generate(model, [{"task": kind, "query": query}], 160)[0]
    spec = parse_spec(reply)
    print(f"\n{kind.upper()}: {'PASS' if spec == expected else 'FAIL'}")
    if spec == expected:
        render(spec)
    else:
        print("Expected:", json.dumps(expected))
        print("Model:", reply)


In [ ]:
tests = [
    (
        "bar", "Books borrowed by genre", "Genre", "Books borrowed",
        ["Science fiction", "Mystery", "Biography", "History"],
        [18, 12, 7, 12],
    ),
    (
        "line", "Tea cooling over time", "Time (min)", "Temperature (°C)",
        [0, 3, 8, 15, 22],
        [82.5, 74, 61.5, 49, 41.5],
    ),
    (
        "pie", "School garden area", "Crop", "Area (m²)",
        ["Tomatoes", "Leafy greens", "Herbs", "Root vegetables"],
        [12.5, 6.25, 2.5, 3.75],
    ),
]

for kind, title, x_label, y_label, x, y in tests:
    print(f"\n{kind.upper()}")
    chart = render({
        "type": kind,
        "title": title,
        "x": {"label": x_label, "values": x},
        "y": {"label": y_label, "values": y},
    })
    if chart is not None:
        print(chart)
